# DeepSeek AI 챗봇 · 아이패드 Colab 실행용
동아대학교 조선해양공학과 진윤석

1. Google Colab에서 이 파일을 열고 Google 계정으로 로그인합니다.
2. 기본 CPU 런타임으로 실행합니다. 아래 1·2·3번 코드 칸의 ▶ 버튼을 차례로 누릅니다.
3. 3번 칸에서 실제 대화 테스트를 확인한 후, 4번 칸에서 자유롭게 대화합니다.
4. `종료`를 입력한 뒤 5번 칸을 실행하면 대화 기록을 다운로드할 수 있습니다.

아이패드 Safari는 입력·출력 화면을 담당하고 모델은 Colab 서버에서 실행됩니다. 최초 설치와 약 1.12GB 모델 다운로드에는 인터넷과 시간이 필요합니다. CPU에서는 답변마다 수십 초 이상 걸릴 수 있습니다. Colab 세션이 종료되면 다시 실행해야 합니다.

모델은 **DeepSeek-R1-Distill-Qwen-1.5B의 GGUF Q4_K_M 버전**입니다. bartowski가 배포한 경량화 가중치를 llama-cpp-python으로 실행합니다. 원본 모델은 DeepSeek에서 공개했습니다.


## 제작 환경의 실제 검증 결과
CPU 4스레드 · GGUF Q4_K_M · llama-cpp-python 0.3.36 · Python 3.12.14

| 시험 | 실제 최종 답변 | 시간 |
| --- | --- | --- |
| 1. What is 7 + 5? Answer in one short sentence. | 7 + 5 equals 12. | 51.66초 |
| 2. 한국어로 한 문장 자기소개해 줘. | 이irteen는 12로 표현된다. | 33.19초 |
| 3. My favorite color is blue. Reply with OK. | OK | 43.14초 |
| 4. What is my favorite color? Answer briefly. | 이어 information이 드리겠습니다. My favorite color is blue. | 40.11초 |

4회 모두 종료 토큰까지 생성됐습니다. 계산 정답과 OK 지시는 충족했습니다. 색 기억 질문에 blue를 포함했지만 혼합 언어와 어색한 표현이 나타났습니다. 한국어 자기소개는 요청을 충족하지 못했습니다. 이 결과는 제작 환경 CPU 실행 결과이며 Colab·iPad Safari 화면에서의 전체 실행은 별도 확인이 필요합니다.


## 1. 라이브러리 설치

In [ ]:
%pip -q install "llama-cpp-python==0.3.36" --only-binary=llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
%pip -q install "huggingface-hub==0.36.2"


## 2. 모델 다운로드와 챗봇 준비

In [ ]:
import os
import json
import time
from datetime import datetime
from pathlib import Path
os.environ["HF_HUB_DISABLE_XET"] = "1"
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

MODEL_REPO = "bartowski/DeepSeek-R1-Distill-Qwen-1.5B-GGUF"
MODEL_FILE = "DeepSeek-R1-Distill-Qwen-1.5B-Q4_K_M.gguf"
model_path = hf_hub_download(MODEL_REPO, MODEL_FILE)
llm = Llama(model_path=model_path, n_ctx=4096, n_threads=4, verbose=False)
records = []
print("DeepSeek 모델 준비 완료 · CPU · Q4_K_M")


def respond(question, history):
    current = history[-6:] + [{"role": "user", "content": question}]
    start = time.perf_counter()
    result = llm.create_chat_completion(
        messages=current, max_tokens=1024, temperature=0.6, top_p=0.95,
    )
    choice = result["choices"][0]
    raw = choice["message"]["content"] or ""
    answer = raw.split("</think>", 1)[-1].strip() if "</think>" in raw else raw.strip()
    seconds = round(time.perf_counter() - start, 2)
    record = {"time": datetime.now().isoformat(), "question": question,
              "answer": answer, "raw_output": raw, "seconds": seconds,
              "finish_reason": choice["finish_reason"], "tokens": result.get("usage", {})}
    records.append(record)
    print("나 >", question)
    print("DeepSeek >", answer)
    print(f"[소요 시간 {seconds}초]\n")
    if choice["finish_reason"] != "stop":
        print("출력 길이 한도에 도달했습니다. 답변이 미완성일 수 있습니다.")
    return current + [{"role": "assistant", "content": answer}]


## 3. 자동 대화 테스트
계산, 한국어 자기소개, 이전 대화 기억을 순서대로 시험합니다. 각 답변 생성에 시간이 걸리므로 완료될 때까지 기다립니다.

In [ ]:
test_history = []
questions = [
    "What is 7 + 5? Answer in one short sentence.",
    "한국어로 한 문장 자기소개해 줘.",
    "My favorite color is blue. Reply with OK.",
    "What is my favorite color? Answer briefly.",
]
for question in questions:
    test_history = respond(question, test_history)


## 4. 자유 대화
질문을 입력하고 Enter를 누릅니다. `새대화`는 문맥 초기화, `종료`는 대화 종료입니다. 결과 저장 전에 종료를 입력합니다.

In [ ]:
history = []
print("DeepSeek 챗봇 시작 | 새대화 / 종료")
while True:
    question = input("질문 입력 > ").strip()
    if question == "종료":
        print("종료했습니다. 5번 칸에서 대화를 저장하세요.")
        break
    if question == "새대화":
        history.clear()
        print("새 대화를 시작합니다.")
        continue
    if not question:
        continue
    if len(question) > 1500:
        print("질문을 1,500자 이내로 입력해 주세요.")
        continue
    try:
        history = respond(question, history)
    except ValueError as error:
        print("대화가 길면 새대화를 입력한 뒤 다시 시도하세요.", str(error))


## 5. 실제 결과 다운로드
TXT는 보고서 작성용이며 JSON에는 원문과 토큰 수, 종료 상태까지 들어 있습니다.

In [ ]:
from google.colab import files
if not records:
    print("먼저 3번 또는 4번 칸을 실행하세요.")
else:
    lines = ["DeepSeek 실제 대화 결과", MODEL_REPO, MODEL_FILE]
    for i, row in enumerate(records, 1):
        lines.extend([f"\n테스트 {i}", "질문 " + row["question"], "답변 " + row["answer"],
                      f"소요 시간 {row['seconds']}초", "종료 상태 " + row["finish_reason"]])
    Path("DeepSeek_results.txt").write_text("\n".join(lines), encoding="utf-8")
    Path("DeepSeek_results.json").write_text(
        json.dumps({"model": MODEL_REPO, "file": MODEL_FILE, "records": records},
                   ensure_ascii=False, indent=2), encoding="utf-8")
    files.download("DeepSeek_results.txt")
    files.download("DeepSeek_results.json")


## 제출 및 해석
실행한 ipynb 파일이 챗봇 프로그램입니다. 실행 결과와 대화 기록을 함께 제출할 수 있습니다. 출력이 생성되는지와 지시·정답을 만족하는지는 각각 확인합니다. 한국어 응답이나 사실 정확도에 한계가 있을 수 있으며, 응답 시간은 컴퓨터 환경에 따라 달라집니다.

## 출처
- 원본 모델 https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B
- 경량화 가중치 https://huggingface.co/bartowski/DeepSeek-R1-Distill-Qwen-1.5B-GGUF
- 실행 라이브러리 https://github.com/abetlen/llama-cpp-python
- Colab 안내 https://research.google.com/colaboratory/faq.html?hl=ko
